# Repeatable YOLO26 NCNN Test on PYNQ-Z2

This notebook calls the `measure_model.sh` interfaces use identical benchmark settings.


In [19]:
# Main measurement: about 20 mins each baseline.
from pathlib import Path
import subprocess
import os

# Change these values for each new model.
MODEL_NAME = "4_layer_replacement_320_FP16"
MODE = "benchmark"   # benchmark, validate, or all
IMAGE_LIMIT = 1      # 1 = smoke test, 0 = full run (50)
IMGSZ = 320

ROOT = Path("/home/xilinx/yolo26_ps")
SCRIPT = ROOT / "1_scripts" / "measure_model.sh"

# measure_model.sh accepts only:
# model_name, mode, image_limit
command = [
    "bash",
    str(SCRIPT),
    MODEL_NAME,
    MODE,
    str(IMAGE_LIMIT),
]

# Pass image resolution separately as an environment variable.
env = os.environ.copy()
env["IMGSZ"] = str(IMGSZ)
print(
    "Running: IMGSZ={} {}".format(
        IMGSZ,
        " ".join(command),
    )
)
print("=" * 70)

result = subprocess.run(
    command,
    cwd=str(ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
)

print(result.stdout)
print("=" * 70)
print("Exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Model test failed. Read the error above."
    )

Running: IMGSZ=320 bash /home/xilinx/yolo26_ps/1_scripts/measure_model.sh 4_layer_replacement_320_FP16 benchmark 1
Model: 4_layer_replacement_320_FP16
Mode: benchmark
Image limit: 1
Model: /home/xilinx/yolo26_ps/1_models/4_layer_replacement_320_FP16
Classes: 11
Images: 1
Mode: benchmark

Warm-up 1/5
Warm-up 2/5
Warm-up 3/5
Warm-up 4/5
Warm-up 5/5
Output tensor: dims=2 w=2100 h=15 c=1

Benchmark 1/3
Benchmark 2/3
Benchmark 3/3
Benchmark completed.
Summary: /home/xilinx/yolo26_ps/2_results/4_layer_replacement_320_FP16/performance.csv
Median compute latency: 2283.92 ms
Compute FPS: 0.423073
Total FPS including image read: 0.418725
Peak RSS: 78.25 MiB
Normalized board CPU: 96.1005%
Results saved under: /home/xilinx/yolo26_ps/2_results/4_layer_replacement_320_FP16
metadata.yaml  668 bytes
model_hashes.txt  427 bytes
model_size.csv  112 bytes
performance.csv  675 bytes
run_config.txt  279 bytes

Key performance results
-----------------------
Model:                    4_layer_replacement_320

In [6]:
# Results checking.
import csv
from pathlib import Path

#MODEL_NAME = "4_layer_replacement"
#ROOT = Path("/home/xilinx/yolo26_ps")

RESULT_DIR = ROOT / "2_results" / MODEL_NAME
PERFORMANCE_CSV = RESULT_DIR / "performance.csv"
MODEL_SIZE_CSV = RESULT_DIR / "model_size.csv"

# Display compact result tables without requiring pandas.
for result_file in [PERFORMANCE_CSV, MODEL_SIZE_CSV]:
    print(f"\n===== {result_file.name} =====")
    if not result_file.is_file():
        print("Not generated in this mode.")
        continue

    with result_file.open(newline="") as file:
        reader = csv.DictReader(file)
        for row in reader:
            for key, value in row.items():
                print(f"{key}: {value}")



===== performance.csv =====
model: 4_layer_replacement
classes: 11
imgsz: 640
threads: 2
warmup: 5
repeat: 3
benchmark_images: 50
measured_inferences: 150
conf: 0.25000000
iou: 0.69999999
max_det: 300
output_dims: 2
output_w: 8400
output_h: 15
output_c: 1
mean_io_ms: 21.65054907
mean_preprocess_ms: 75.10764547
mean_inference_ms: 7958.01673463
mean_postprocess_ms: 1.83627833
mean_compute_ms: 8034.96065842
median_compute_ms: 7869.81364350
p95_compute_ms: 8745.30935110
compute_fps: 0.12445612
mean_total_ms: 8056.61120749
median_total_ms: 7891.19952250
p95_total_ms: 8761.31949905
total_fps: 0.12407486
peak_rss_mib: 106.10546875
raw_process_cpu_percent: 187.42036862
normalized_board_cpu_percent: 93.71018431
logical_cores: 2

===== model_size.csv =====
model: 4_layer_replacement
param_bytes: 26382
bin_bytes: 8582536
total_bytes: 8608918
total_mib: 8.21010399


In [8]:
# Show every generated file and the archive path.
print(f"Result directory: {RESULT_DIR}")
for path in sorted(RESULT_DIR.glob("*")):
    if path.is_file():
        print(f"- {path.name}: {path.stat().st_size} bytes")


Result directory: /home/xilinx/yolo26_ps/2_results/4_layer_replacement
- metadata.yaml: 670 bytes
- model_hashes.txt: 400 bytes
- model_size.csv: 103 bytes
- performance.csv: 670 bytes
- power.csv: 128 bytes
- run_config.txt: 270 bytes


In [20]:
# Zip result files
from pathlib import Path
import zipfile

# Edit needed models
MODEL_FOLDERS = [
#     "0_baseline_official",
#     "1_baseline_light",
#     "2_baseline_full",
#     "2_baseline_full_320",
#     "2_baseline_full_480",
#     "2_baseline_full_FP16",
#     "3_global_L1",
#     "3_global_L1_snow",
#     "4_layer_replacement",
    "4_layer_replacement_320_FP16",
#     "4_layer_replacement_snow",
]

# Files needed for each model.
REPORT_FILES = [
    "performance.csv",
    "model_size.csv",
    "run_config.txt",
    "metadata.yaml",
    "model_hashes.txt",
]

# Automatically locate 2_results from the current Jupyter directory.
current = Path.cwd().resolve()

candidates = [
    current,
    current / "2_results",
    current.parent,
    current.parent / "2_results",
    Path("/home/xilinx/yolo26_ps/2_results"),
    Path("/home/xilinx/yolo26_ps_sc/2_results"),
]

RESULTS_DIR = None

for candidate in candidates:
    if candidate.is_dir() and candidate.name == "2_results":
        RESULTS_DIR = candidate
        break

if RESULTS_DIR is None:
    raise RuntimeError(
        "Could not locate the 2_results directory. "
        "Open this notebook from the project or 2_results folder."
    )

# The same ZIP name is replaced each time the cell is run.
ZIP_PATH = RESULTS_DIR / "pynq_report_results.zip"

included = []
skipped = []

with zipfile.ZipFile(
    str(ZIP_PATH),
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for model_name in MODEL_FOLDERS:
        model_dir = RESULTS_DIR / model_name

        # Ignore a model folder when it does not exist.
        if not model_dir.is_dir():
            skipped.append(str(model_dir))
            continue

        for filename in REPORT_FILES:
            source = model_dir / filename

            # Include only files that exist and are not empty.
            if source.is_file() and source.stat().st_size > 0:
                archive.write(
                    str(source),
                    arcname="{}/{}".format(
                        model_name,
                        filename,
                    ),
                )
                included.append(str(source))
            else:
                skipped.append(str(source))

print("Results directory:")
print(RESULTS_DIR)

print("\nZIP created:")
print(ZIP_PATH)

print("\nIncluded files:", len(included))
print("Skipped missing or empty files:", len(skipped))

if skipped:
    print("\nSkipped:")
    for path in skipped:
        print("-", path)

Results directory:
/home/xilinx/yolo26_ps/2_results

ZIP created:
/home/xilinx/yolo26_ps/2_results/pynq_report_results.zip

Included files: 5
Skipped missing or empty files: 0
